In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

# load dataset
df = pd.read_csv('modular 1.csv', encoding='latin1')

# view 5 rows datasets
df.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# SECTION 1 — DATA LOADING & EXPLORATION


# 1.1 Load dataset
df = pd.read_csv('modular 1.csv', encoding='latin1')

# Convert date column
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("=" * 55)
print("       GENERAL DATASET OVERVIEW")
print("=" * 55)
print(f"  Dimensions        : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Time period       : {df['InvoiceDate'].min()}  →  {df['InvoiceDate'].max()}")
print(f"  Unique customers  : {df['CustomerID'].nunique():,}")
print(f"  Unique products   : {df['StockCode'].nunique():,}")
print(f"  Unique invoices   : {df['InvoiceNo'].nunique():,}")
print(f"  Countries covered : {df['Country'].nunique()}")
print("=" * 55)

# 1.2 Data types & missing values
print("\n── Column Types & Missing Values ──")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
info_df = pd.DataFrame({
    'Type'       : df.dtypes,
    'Missing'    : missing,
    '% Missing'  : missing_pct
})
print(info_df)

# 1.3 Descriptive statistics
print("\n── Descriptive Statistics (Numerical Columns) ──")
print(df[['Quantity', 'UnitPrice']].describe().round(2))

# ── 1.4 Exploratory Visualizations
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

monthly_orders = df.groupby('YearMonth')['InvoiceNo'].nunique().reset_index()
monthly_orders['YearMonth'] = monthly_orders['YearMonth'].astype(str)

top10_countries = (df[df['Country'] != 'United Kingdom']
                   .groupby('Country')['InvoiceNo']
                   .nunique()
                   .sort_values(ascending=False)
                   .head(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Section 1 — General Exploration', fontsize=14, fontweight='bold')

# Plot 1: Monthly order volume
axes[0].plot(monthly_orders['YearMonth'],
             monthly_orders['InvoiceNo'],
             marker='o', linewidth=2)
axes[0].set_title('Monthly Order Volume')
axes[0].set_xlabel('Month')
axes[0].set_ylabel("Number of Invoices")
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Plot 2: Top 10 countries (excluding UK)
axes[1].barh(top10_countries.index[::-1],
             top10_countries.values[::-1])
axes[1].set_title('Top 10 Countries (excluding UK)')
axes[1].set_xlabel("Number of Invoices")
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('section1_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Section 1 completed — visualization saved.")

In [ ]:
# ─────────────────────────────────────────
# SECTION 2 — DATA PREPROCESSING & CLEANING
# ─────────────────────────────────────────

print("=" * 55)
print("     SECTION 2 — DATA CLEANING")
print("=" * 55)

# ── 2.1 Working copy
df_clean = df.copy()

print(f"\n  Initial size : {df_clean.shape[0]:,} rows")

# ── 2.2 Remove missing values
before = len(df_clean)
df_clean.dropna(subset=['CustomerID'], inplace=True)
after_na = len(df_clean)
print(f"  Rows removed (missing CustomerID) : {before - after_na:,}")

# ── 2.3 Remove duplicates
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
after_dup = len(df_clean)
print(f"  Duplicates removed                : {before - after_dup:,}")

# ── 2.4 Filter invalid quantities and prices
before = len(df_clean)
df_clean = df_clean[df_clean['Quantity']  > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]
after_neg = len(df_clean)
print(f"  Rows removed (Qty/Price ≤ 0)      : {before - after_neg:,}")

# ── 2.5 Remove cancelled invoices (C...)
before = len(df_clean)
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
after_cancel = len(df_clean)
print(f"  Cancelled invoices removed        : {before - after_cancel:,}")

# ── 2.6 Remove non-product StockCodes
before = len(df_clean)
invalid_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
df_clean = df_clean[~df_clean['StockCode'].astype(str).isin(invalid_codes)]
after_code = len(df_clean)
print(f"  Invalid stock codes removed       : {before - after_code:,}")

# ── 2.7 Feature engineering
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['CustomerID']  = df_clean['CustomerID'].astype(int).astype(str)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['Year']        = df_clean['InvoiceDate'].dt.year
df_clean['Month']       = df_clean['InvoiceDate'].dt.month
df_clean['DayOfWeek']   = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour']        = df_clean['InvoiceDate'].dt.hour

print(f"\n   Final size after cleaning : {df_clean.shape[0]:,} rows")
print(f"  Remaining customers         : {df_clean['CustomerID'].nunique():,}")
print(f"  Remaining products          : {df_clean['StockCode'].nunique():,}")

# ── 2.8 Before / After summary
print("\n── Before / After Cleaning Summary ──")
summary = pd.DataFrame({
    'Step'    : ['Initial', 'No missing CustomerID',
                 'No duplicates', 'No Qty/Price ≤ 0',
                 'No cancellations', 'No invalid codes'],
    'Rows'    : [df.shape[0], after_na, after_dup,
                 after_neg, after_cancel, after_code]
})
summary['Rows Removed'] = summary['Rows'].shift(1).fillna(df.shape[0]).astype(int) - summary['Rows']
print(summary.to_string(index=False))

# ── 2.9 Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Section 2 — Post-Cleaning Analysis', fontsize=14, fontweight='bold')

# Plot 1: Total amount distribution
axes[0].hist(df_clean['TotalAmount'].clip(upper=500),
             bins=60, edgecolor='white')
axes[0].set_title('Distribution of transaction amount\n(capped at 500)')
axes[0].set_xlabel('TotalAmount (£)')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Plot 2: Orders by day of the week
order_days = df_clean.groupby('DayOfWeek')['InvoiceNo'].nunique()
day_order  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Sunday']
order_days = order_days.reindex(day_order)
axes[1].bar(order_days.index, order_days.values, edgecolor='white')
axes[1].set_title('Orders by Day of the Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel("Number of Invoices")
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# Plot 3: Orders by hour
order_hours = df_clean.groupby('Hour')['InvoiceNo'].nunique()
axes[2].bar(order_hours.index, order_hours.values, edgecolor='white')
axes[2].set_title("Orders by Hour of the Day")
axes[2].set_xlabel('Hour')
axes[2].set_ylabel("Number of Invoices")
axes[2].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('section2_cleaning.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2.10 Save cleaned dataset
df_clean.to_csv('modular1_clean.csv', index=False)
print("\n Section 2 completed — cleaned dataset saved as 'modular1_clean.csv'")

In [ ]:
# ─────────────────────────────────────────────────────
# SECTION 3 — CUSTOMER SEGMENTATION (RFM + K-MEANS)
# ─────────────────────────────────────────────────────

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from mpl_toolkits.mplot3d import Axes3D

print("=" * 55)
print("   SECTION 3 — CUSTOMER SEGMENTATION (RFM + K-MEANS)")
print("=" * 55)

# ── 3.1 RFM table construction
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',  lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',    'nunique'),
    Monetary  = ('TotalAmount',  'sum')
).reset_index()

print("\n── RFM table preview (first 5 rows) ──")
print(rfm.head().to_string(index=False))
print(f"\n  Customers in RFM table : {len(rfm):,}")

print("\n── RFM statistics ──")
print(rfm[['Recency','Frequency','Monetary']].describe().round(2))

# ── 3.2 Outlier treatment (IQR method)
def cap_outliers(df, col):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)
    return df

for col in ['Recency', 'Frequency', 'Monetary']:
    rfm = cap_outliers(rfm, col)

# ── 3.3 Normalization
scaler   = StandardScaler()
rfm_cols = ['Recency', 'Frequency', 'Monetary']
rfm_scaled = scaler.fit_transform(rfm[rfm_cols])

# ── 3.4 Elbow method + Silhouette score
inertia    = []
sil_scores = []
K_range    = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Section 3 — Optimal number of clusters", fontsize=13, fontweight='bold')

axes[0].plot(K_range, inertia, marker='o', linewidth=2)
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters K")
axes[0].set_ylabel("Inertia")
axes[0].grid(linestyle='--', alpha=0.5)

axes[1].plot(K_range, sil_scores, marker='s', linewidth=2)
axes[1].set_title("Silhouette Score by K")
axes[1].set_xlabel("Number of clusters K")
axes[1].set_ylabel("Silhouette Score")
axes[1].grid(linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('section3_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3.5 Final K-Means model (k=4)
OPTIMAL_K = 4
km_final  = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled)

print(f"\n  Silhouette Score (K={OPTIMAL_K}) : "
      f"{silhouette_score(rfm_scaled, rfm['Cluster']):.4f}")

# ── 3.6 Cluster profiling
cluster_profile = rfm.groupby('Cluster')[rfm_cols].mean().round(2)
cluster_profile['Num_Customers'] = rfm.groupby('Cluster')['CustomerID'].count()

def label_cluster(row):
    if row['Monetary'] >= cluster_profile['Monetary'].median() and \
       row['Frequency'] >= cluster_profile['Frequency'].median():
        return 'Champions'
    elif row['Recency'] <= cluster_profile['Recency'].median() and \
         row['Frequency'] >= cluster_profile['Frequency'].median():
        return 'Loyal Customers'
    elif row['Recency'] >= cluster_profile['Recency'].quantile(0.75):
        return 'Inactive'
    else:
        return 'At Risk'

cluster_profile['Segment'] = cluster_profile.apply(label_cluster, axis=1)

print("\n── Cluster profile ──")
print(cluster_profile.to_string())

# ── 3.7 Visualizations
colors = ['steelblue', 'coral', 'mediumseagreen', 'gold']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Section 3 — Customer Segmentation', fontsize=13, fontweight='bold')

for c in range(OPTIMAL_K):
    subset = rfm[rfm['Cluster'] == c]
    axes[0].scatter(subset['Recency'], subset['Monetary'],
                    label=f"Cluster {c}", alpha=0.5, s=15)
axes[0].set_title('Recency vs Monetary')
axes[0].set_xlabel('Recency (days)')
axes[0].set_ylabel('Monetary (£)')
axes[0].legend()
axes[0].grid(linestyle='--', alpha=0.4)

for c in range(OPTIMAL_K):
    subset = rfm[rfm['Cluster'] == c]
    axes[1].scatter(subset['Frequency'], subset['Monetary'],
                    label=f"Cluster {c}", alpha=0.5, s=15)
axes[1].set_title('Frequency vs Monetary')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Monetary (£)')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.4)

seg_counts = rfm['Cluster'].value_counts().sort_index()
axes[2].bar([f"Cluster {i}" for i in seg_counts.index],
            seg_counts.values)
axes[2].set_title('Number of customers per cluster')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Customers')
axes[2].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('section3_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3.8 Save
rfm.to_csv('modular1_rfm.csv', index=False)
print("\n Section 3 completed — RFM table saved as 'modular1_rfm.csv'")

In [ ]:
# ─────────────────────────────────────────────────────
# SECTION 4 — MARKET BASKET ANALYSIS (APRIORI)
# ─────────────────────────────────────────────────────

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("=" * 55)
print("   SECTION 4 — MARKET BASKET ANALYSIS (APRIORI)")
print("=" * 55)

# ── 4.1 Basket preparation (UK only)
basket_df = df_clean[df_clean['Country'] == 'United Kingdom'].copy()

print(f"\n  UK Transactions   : {basket_df['InvoiceNo'].nunique():,}")
print(f"  Unique products   : {basket_df['StockCode'].nunique():,}")

# ── 4.2 Basket matrix (Invoice x Product)
basket_matrix = (
    basket_df
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack(fill_value=0)
)

# Binary encoding
def encode_units(x):
    return 1 if x >= 1 else 0

basket_encoded = basket_matrix.applymap(encode_units)

print(f"\n  Basket matrix shape : {basket_encoded.shape}")
print(f"  (Invoices × Products)")

# ── 4.3 Apriori algorithm
print("\n   Computing frequent itemsets (min_support=0.02)...")

frequent_itemsets = apriori(
    basket_encoded,
    min_support=0.02,
    use_colnames=True,
    max_len=3
)

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print(f"\n  Frequent itemsets found : {len(frequent_itemsets):,}")

print("\n── Top 10 frequent itemsets ──")
top_itemsets = (frequent_itemsets[frequent_itemsets['length'] >= 2]
                .sort_values('support', ascending=False)
                .head(10))

top_itemsets_display = top_itemsets.copy()
top_itemsets_display['itemsets'] = top_itemsets_display['itemsets'].apply(
    lambda x: ', '.join(list(x)))
top_itemsets_display['support'] = top_itemsets_display['support'].round(4)

print(top_itemsets_display[['itemsets','support','length']].to_string(index=False))

# ── 4.4 Association rules
rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=1.5
)

rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"\n  Rules generated (lift ≥ 1.5) : {len(rules):,}")

rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

print("\n── Top 15 rules ──")
display_cols = ['antecedents_str', 'consequents_str',
                'support', 'confidence', 'lift']
print(rules[display_cols].head(15).round(4).to_string(index=False))

# ── 4.5 Strong rules
strong_rules = rules[
    (rules['confidence'] >= 0.5) &
    (rules['lift'] >= 2.0)
].copy()

print(f"\n  Strong rules : {len(strong_rules):,}")

print("\n── Top 10 strong rules ──")
print(strong_rules[display_cols].head(10).round(4).to_string(index=False))

# ── 4.6 Visualizations
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
fig.suptitle('Section 4 — Market Basket Analysis', fontsize=13, fontweight='bold')

sc = axes[0].scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    cmap='RdYlGn',
    alpha=0.6, s=40
)
plt.colorbar(sc, ax=axes[0], label='Lift')
axes[0].set_title('Support vs Confidence (color = Lift)')
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Confidence')

axes[1].hist(rules['lift'], bins=40, edgecolor='white')
axes[1].set_title('Lift Distribution')

top10_rules = rules.head(10).copy()
top10_rules['rule_label'] = (
    top10_rules['antecedents_str'].str[:25] + ' →\n' +
    top10_rules['consequents_str'].str[:25]
)

axes[2].barh(top10_rules['rule_label'][::-1],
             top10_rules['lift'][::-1])
axes[2].set_title('Top 10 Rules by Lift')

plt.tight_layout()
plt.savefig('section4_apriori.png', dpi=150)
plt.show()

# ── 4.7 Save rules
rules_save = rules[['antecedents_str', 'consequents_str',
                    'support', 'confidence', 'lift',
                    'leverage', 'conviction']].copy()

rules_save.to_csv('modular1_rules.csv', index=False)

print("\n Section 4 completed — rules saved as 'modular1_rules.csv'")

In [ ]:
# ─────────────────────────────────────────────────────
# SECTION 5 — CONCLUSION & RESULTS SUMMARY
# ─────────────────────────────────────────────────────

print("=" * 60)
print("      SECTION 5 — CONCLUSION & SUMMARY")
print("=" * 60)

# ── SAFETY CHECKS (important if sections run separately)
if 'Nb_Clients' not in cluster_profile.columns:
    cluster_profile['Nb_Clients'] = rfm.groupby('Cluster')['CustomerID'].count()

# ── 5.1 Full pipeline recap
print("""
╔══════════════════════════════════════════════════════════╗
║              DATA MINING PIPELINE — SUMMARY              ║
╠══════════════════════════════════════════════════════════╣
║  Section 1 │ Exploration      │ Patterns & trends        ║
║  Section 2 │ Data Cleaning    │ High-quality dataset     ║
║  Section 3 │ RFM Segmentation │ 4 customer segments      ║
║  Section 4 │ Apriori (MBA)    │ Association rules        ║
╚══════════════════════════════════════════════════════════╝
""")

# ── 5.2 Key project metrics ─
print("── Key Project Metrics ──")

total_revenue    = df_clean['TotalAmount'].sum()
avg_order_value  = df_clean.groupby('InvoiceNo')['TotalAmount'].sum().mean()
best_country     = df_clean.groupby('Country')['TotalAmount'].sum().idxmax()
top_product      = df_clean.groupby('Description')['TotalAmount'].sum().idxmax()
best_cluster     = cluster_profile['Monetary'].idxmax()

print(f"""
   Dataset
     • Transactions analyzed  : {df_clean.shape[0]:>10,}
     • Unique customers       : {df_clean['CustomerID'].nunique():>10,}
     • Unique products        : {df_clean['StockCode'].nunique():>10,}
     • Countries covered      : {df_clean['Country'].nunique():>10,}

   Business Performance
     • Total revenue          : £{total_revenue:>12,.2f}
     • Average order value    : £{avg_order_value:>12,.2f}
     • Top country (revenue)  : {best_country}
     • Top product (revenue)  : {top_product[:45]}

   RFM Segmentation (K-Means, K=4)
     • Silhouette Score       : {silhouette_score(rfm_scaled, rfm['Cluster']):.4f}
     • Most valuable cluster  : Cluster {best_cluster}
     • Customers segmented    : {len(rfm):,}

   Market Basket Analysis (Apriori)
     • Frequent itemsets      : {len(frequent_itemsets):>10,}
     • Rules generated        : {len(rules):>10,}
     • Strong rules           : {len(strong_rules):>10,}
     • Best Lift              : {rules['lift'].max():>10.4f}
     • Best Confidence        : {rules['confidence'].max():>10.4f}
""")

# ── 5.3 Customer segment insights
print("── Customer Segment Summary ──\n")

for i, row in cluster_profile.iterrows():
    print(f"  Cluster {i} — {row['Segment']}")
    print(f"    Recency   : {row['Recency']:.0f} days")
    print(f"    Frequency : {row['Frequency']:.1f} orders")
    print(f"    Monetary  : £{row['Monetary']:,.2f}")
    print(f"    Customers : {int(row['Nb_Clients']):,}")

    if '' in row['Segment']:
        recommendation = "Retain with VIP programs and exclusive offers"
    elif '' in row['Segment']:
        recommendation = "Maintain engagement through targeted newsletters"
    elif '' in row['Segment']:
        recommendation = "Run reactivation campaigns with attractive discounts"
    else:
        recommendation = "Apply progressive nurturing and personalized offers"

    print(f"     Action : {recommendation}\n")

# ── 5.4 Top 5 association rules ─
print("── Top 5 Association Rules (Product Recommendations) ──\n")

for i, row in rules.head(5).iterrows():
    print(f"  Rule {i+1} :")
    print(f"    IF     : {row['antecedents_str'][:50]}")
    print(f"    THEN   : {row['consequents_str'][:50]}")
    print(f"    Support    = {row['support']:.4f} | "
          f"Confidence = {row['confidence']:.4f} | "
          f"Lift = {row['lift']:.4f}\n")

# ── 5.5 Final dashboard
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Section 5 — Final Dashboard\nCustomer Segmentation & Market Basket Analysis',
             fontsize=14, fontweight='bold', y=0.98)

# Graph 1: Monthly revenue
ax1 = fig.add_subplot(2, 3, 1)
monthly_revenue = (df_clean.groupby(df_clean['InvoiceDate']
                   .dt.to_period('M'))['TotalAmount']
                   .sum().reset_index())
monthly_revenue['InvoiceDate'] = monthly_revenue['InvoiceDate'].astype(str)

ax1.fill_between(range(len(monthly_revenue)),
                 monthly_revenue['TotalAmount'], alpha=0.4)
ax1.plot(range(len(monthly_revenue)),
         monthly_revenue['TotalAmount'], linewidth=2)

ax1.set_xticks(range(len(monthly_revenue)))
ax1.set_xticklabels(monthly_revenue['InvoiceDate'],
                    rotation=45, fontsize=6)
ax1.set_title("Monthly Revenue (£)", fontweight='bold')
ax1.set_ylabel("£")
ax1.grid(axis='y', linestyle='--', alpha=0.4)

# Graph 2: Segment distribution
ax2 = fig.add_subplot(2, 3, 2)
seg_sizes  = cluster_profile['Nb_Clients'].values
seg_labels = [f"Cluster {i}" for i in cluster_profile.index]

ax2.pie(seg_sizes, labels=seg_labels, autopct='%1.1f%%', startangle=90)
ax2.set_title("Customer Segments Distribution", fontweight='bold')

# Graph 3: Monetary per cluster
ax3 = fig.add_subplot(2, 3, 3)
ax3.bar([f"C{i}" for i in cluster_profile.index],
        cluster_profile['Monetary'])
ax3.set_title("Average Monetary per Cluster (£)", fontweight='bold')

# Graph 4: Top products
ax4 = fig.add_subplot(2, 3, 4)
top10_prod = (df_clean.groupby('Description')['TotalAmount']
              .sum().sort_values(ascending=False).head(10))
ax4.barh([p[:30] for p in top10_prod.index[::-1]],
         top10_prod.values[::-1])
ax4.set_title("Top 10 Products by Revenue", fontweight='bold')

# Graph 5: Rules scatter
ax5 = fig.add_subplot(2, 3, 5)
ax5.scatter(rules['support'], rules['lift'], alpha=0.6)
ax5.set_title("Association Rules: Support vs Lift", fontweight='bold')

# Graph 6: RFM clusters
ax6 = fig.add_subplot(2, 3, 6)
for c in range(OPTIMAL_K):
    subset = rfm[rfm['Cluster'] == c]
    ax6.scatter(subset['Recency'], subset['Frequency'], label=f"C{c}")

ax6.set_title("Recency vs Frequency by Cluster", fontweight='bold')
ax6.legend()

plt.tight_layout()
plt.savefig('section5_dashboard_final.png', dpi=150)
plt.show()

# ── 5.6 Save text report ─
with open('final_report.txt', 'w', encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("FINAL REPORT — CUSTOMER SEGMENTATION & MBA\n")
    f.write("Gountante Douti | MSc Statistics\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Transactions analyzed : {df_clean.shape[0]:,}\n")
    f.write(f"Customers             : {df_clean['CustomerID'].nunique():,}\n")
    f.write(f"Total revenue         : £{total_revenue:,.2f}\n")
    f.write(f"Clusters              : {OPTIMAL_K}\n")
    f.write(f"Association rules     : {len(rules):,}\n")

print("\n Section 5 completed — report saved as 'final_report.txt'")
print("\n PROJECT COMPLETED SUCCESSFULLY!")

In [ ]:

# SECTION 5 — FAST VERSION (OPTIMIZED)


print("=" * 60)
print("      SECTION 5 — CONCLUSION & SUMMARY (FAST)")
print("=" * 60)

# ── 5.1 Safety Fix
if 'Nb_Clients' not in cluster_profile.columns:
    cluster_profile['Nb_Clients'] = rfm.groupby('Cluster')['CustomerID'].count()

# ── 5.2 PRE-COMPUTE (IMPORTANT )
total_revenue   = df_clean['TotalAmount'].sum()
avg_order_value = df_clean.groupby('InvoiceNo')['TotalAmount'].sum().mean()
best_country    = df_clean.groupby('Country')['TotalAmount'].sum().idxmax()
top_product     = df_clean.groupby('Description')['TotalAmount'].sum().idxmax()

monthly_revenue = df_clean.groupby(df_clean['InvoiceDate'].dt.to_period('M'))['TotalAmount'].sum()
top10_prod      = df_clean.groupby('Description')['TotalAmount'].sum().nlargest(10)

# ── 5.3 PRINT METRICS (FAST)
print(f"""
Transactions   : {df_clean.shape[0]:,}
Customers      : {df_clean['CustomerID'].nunique():,}
Revenue        : £{total_revenue:,.2f}
Top Country    : {best_country}
Top Product    : {top_product[:40]}
""")

# ── 5.4 LIGHT DASHBOARD (3 graphs only )
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Graph 1: Monthly revenue
axes[0].plot(monthly_revenue.values)
axes[0].set_title("Monthly Revenue")

# Graph 2: Segment distribution
axes[1].pie(cluster_profile['Nb_Clients'], autopct='%1.1f%%')
axes[1].set_title("Customer Segments")

# Graph 3: Top products
axes[2].barh([p[:20] for p in top10_prod.index[::-1]],
             top10_prod.values[::-1])
axes[2].set_title("Top Products")

plt.tight_layout()
plt.savefig('section5_fast.png', dpi=120)
plt.show()

# ── 5.5 SAVE REPORT (LIGHT)
with open('final_report.txt', 'w') as f:
    f.write("FINAL SUMMARY\n")
    f.write(f"Revenue: £{total_revenue:,.2f}\n")
    f.write(f"Customers: {df_clean['CustomerID'].nunique():,}\n")

print("\n Section 5 completed (FAST)")

In [ ]:
# ─────────────────────────────────────────────────────
# SECTION 6 — CLASSIFICATION
# ─────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("=" * 60)
print("      SECTION 6 — CLASSIFICATION MODELS (FAST)")
print("=" * 60)

# ── 6.1 Features & Target
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Cluster']

# ── 6.2 Train/Test Split ─
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining size : {X_train.shape[0]:,}")
print(f"Testing size  : {X_test.shape[0]:,}")

# ── 6.3 Logistic Regression
print("\n Running Logistic Regression...")

log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

acc_log = accuracy_score(y_test, y_pred_log)

print("\n── Logistic Regression Results ──")
print(f"Accuracy : {acc_log:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

# Confusion Matrix — Logistic
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_log),
            annot=True, fmt='d')
plt.title("Confusion Matrix — Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig('section6_logistic_cm.png', dpi=150)
plt.show()

# ── 6.4 Random Forest
print("\n Running Random Forest (optimized)...")

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)

print("\n── Random Forest Results ──")
print(f"Accuracy : {acc_rf:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

# Confusion Matrix — Random Forest
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf),
            annot=True, fmt='d')
plt.title("Confusion Matrix — Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig('section6_rf_cm.png', dpi=150)
plt.show()

# ── 6.5 Feature Importance
importance = rf_model.feature_importances_

plt.figure(figsize=(6,4))
plt.bar(X.columns, importance)
plt.title("Feature Importance (Random Forest)")
plt.ylabel("Importance")
plt.tight_layout()
plt.savefig('section6_feature_importance.png', dpi=150)
plt.show()

# ── 6.6 Model Comparison
print("\n── Model Comparison ──")
print(f"Logistic Regression Accuracy : {acc_log:.4f}")
print(f"Random Forest Accuracy       : {acc_rf:.4f}")

best_model = "Random Forest" if acc_rf > acc_log else "Logistic Regression"
print(f"\n Best Model : {best_model}")

# ── 6.7 Accuracy Comparison Plot (NEW )
plt.figure(figsize=(6,4))
plt.bar(['Logistic', 'Random Forest'], [acc_log, acc_rf])
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.savefig("section6_model_comparison.png", dpi=150)
plt.show()

# ── 6.8 Save Results
results = X_test.copy()
results['Actual'] = y_test.values
results['Predicted_RF'] = y_pred_rf
results['Predicted_Log'] = y_pred_log

results.to_csv('modular1_classification_results.csv', index=False)

print("\n Section 6 completed — results saved as 'modular1_classification_results.csv'")